# AMADEUS on Google Colab

Run a saved AMADEUS configuration on a Google Colab GPU.

1. Configure foreground segmentation and save the tracking settings in the desktop application.
2. Open **Colab (beta)**, select the configuration and Google Drive destination, and choose **Prepare for Google Colab**.
3. Choose **Open Google Colab**. The generated notebook includes the configuration path.
4. Check that the Colab runtime has a GPU, then select **Run all**. Authorize access to Google Drive when prompted.

Use a Google account with access to the selected Drive folder. Final CSV files, result videos, required trained weights, summaries, and logs are saved to Google Drive.

See the [online manual](https://amadeus.jpmyrmecol.com/Manual_EN.html#section-10) for the desktop preparation steps.


In [ ]:
CONFIG_PATH = ""  # @param {type:"string"}
# The desktop Prepare step fills this value in the Drive launcher.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

if not CONFIG_PATH.strip():
    raise ValueError('The prepared AMADEUS launcher has no CONFIG_PATH. Prepare the project again.')
if not CONFIG_PATH.replace('\\', '/').startswith('/content/drive/MyDrive'):
    raise ValueError('CONFIG_PATH must start with /content/drive/MyDrive.')
print(f'Using Drive config: {CONFIG_PATH}')

In [ ]:
import platform
import shutil
import subprocess
import sys

nvidia_smi = shutil.which('nvidia-smi')
if nvidia_smi:
    subprocess.run([nvidia_smi], check=False)
else:
    print('nvidia-smi is unavailable; relying on the PyTorch CUDA check.')
import torch
print(f'Python: {platform.python_version()}')
print(f'Torch: {torch.__version__}')
print(f'Torch CUDA build: {torch.version.cuda}')
print(f'CUDA available: {torch.cuda.is_available()}')
if not torch.cuda.is_available():
    raise RuntimeError('GPU runtime is required. In Colab, select Runtime -> Change runtime type -> GPU, reconnect, and run all again.')
print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import hashlib
import json
import os
import shutil
import tempfile
from pathlib import Path, PurePosixPath

RUNTIME_SOURCE = Path(CONFIG_PATH).resolve().parent / 'amadeus_runtime'
REPO_DIR = Path('/content/AMADEUS')
RUNTIME_MANIFEST_NAME = 'amadeus_runtime_manifest.json'
RUNTIME_NOTEBOOK_PATH = 'colab/AMADEUS_Colab.ipynb'
RUNTIME_TEMPLATE_PATH = 'colab/AMADEUS_Colab.template.ipynb'
RUNTIME_REQUIRED_FILES = ('VERSION', 'UV_VERSION', 'uv.lock', RUNTIME_TEMPLATE_PATH)
RUNTIME_PROVENANCE_FIELDS = ('source_ref', 'source_revision', 'version', 'lock_sha256', 'template_sha256', 'package_manifest_sha256')

def runtime_failure(message):
    raise RuntimeError(f'Prepared AMADEUS runtime package {message}.')

def sha256_file(path):
    digest = hashlib.sha256()
    try:
        with path.open('rb') as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(chunk)
    except OSError as exc:
        runtime_failure(f'cannot read {path}: {exc}')
    return digest.hexdigest()

def safe_relative_path(value):
    if not isinstance(value, str) or not value or '\x00' in value:
        runtime_failure(f'contains an invalid file path: {value!r}')
    if value.startswith(('/', '\\')) or '\\' in value or ':' in value:
        runtime_failure(f'contains an unsafe file path: {value!r}')
    path = PurePosixPath(value)
    if path.is_absolute() or path.as_posix() != value or any(part in ('', '.', '..') for part in path.parts):
        runtime_failure(f'contains an unsafe file path: {value!r}')
    return value

def package_path(root, relative):
    current = root
    for part in relative.split('/'):
        current = current / part
        if current.is_symlink():
            runtime_failure(f'contains a symlinked path: {current}')
    return current

def canonical_manifest_hash(entries):
    payload = json.dumps(entries, ensure_ascii=False, sort_keys=True, separators=(',', ':')).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()

def validate_and_stage_runtime(source, destination):
    if source.is_symlink() or not source.is_dir():
        runtime_failure(f'is missing or is not a directory: {source}')
    manifest_path = source / RUNTIME_MANIFEST_NAME
    if manifest_path.is_symlink() or not manifest_path.is_file():
        runtime_failure(f'manifest is missing: {manifest_path}')
    try:
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    except (OSError, UnicodeError, json.JSONDecodeError) as exc:
        runtime_failure(f'manifest cannot be read: {manifest_path} ({exc})')
    if not isinstance(manifest, dict):
        runtime_failure('manifest root is not an object')
    if manifest.get('schema_version') != 2 or manifest.get('package_name') != 'amadeus-colab-runtime' or manifest.get('package_root') != 'amadeus_runtime':
        runtime_failure('manifest identity is unexpected')
    for field in ('source_ref', 'source_revision', 'version'):
        if not isinstance(manifest.get(field), str) or not manifest[field].strip():
            runtime_failure(f'manifest {field} is missing')
    raw_entries = manifest.get('files')
    if not isinstance(raw_entries, list) or not raw_entries:
        runtime_failure('manifest has no file entries')
    entries = []
    seen = set()
    for index, raw_entry in enumerate(raw_entries):
        if not isinstance(raw_entry, dict):
            runtime_failure(f'manifest file entry {index} is not an object')
        relative = safe_relative_path(raw_entry.get('path'))
        if relative.casefold() in seen or relative.casefold() in {RUNTIME_MANIFEST_NAME.casefold(), RUNTIME_NOTEBOOK_PATH.casefold()}:
            runtime_failure(f'manifest contains a duplicate or reserved file path: {relative}')
        seen.add(relative.casefold())
        size = raw_entry.get('size')
        digest = raw_entry.get('sha256')
        if isinstance(size, bool) or not isinstance(size, int) or size < 0 or not isinstance(digest, str) or len(digest) != 64 or any(character not in '0123456789abcdef' for character in digest):
            runtime_failure(f'manifest has invalid metadata for {relative}')
        entries.append({'path': relative, 'size': size, 'sha256': digest})
    if manifest.get('package_manifest_sha256') != canonical_manifest_hash(entries):
        runtime_failure('manifest file-entry hash does not match package_manifest_sha256')
    entry_by_path = {entry['path']: entry for entry in entries}
    for relative in RUNTIME_REQUIRED_FILES:
        if relative not in entry_by_path:
            runtime_failure(f'manifest is missing required file {relative}')
    mutable_files = manifest.get('mutable_files')
    expected_mutable_files = [{'path': RUNTIME_NOTEBOOK_PATH, 'kind': 'project-specific-launcher', 'template_path': RUNTIME_TEMPLATE_PATH}]
    if mutable_files != expected_mutable_files:
        runtime_failure('manifest mutable_files is invalid')
    launcher_path = package_path(source, RUNTIME_NOTEBOOK_PATH)
    if not launcher_path.is_file():
        runtime_failure(f'mutable launcher is missing: {launcher_path}')
    allowed_files = set(entry_by_path) | {RUNTIME_MANIFEST_NAME, RUNTIME_NOTEBOOK_PATH}
    for current, directory_names, file_names in os.walk(source, topdown=True, followlinks=False):
        current_path = Path(current)
        for name in list(directory_names):
            child = current_path / name
            relative = child.relative_to(source).as_posix()
            if child.is_symlink() or not any(path == relative or path.startswith(relative + '/') for path in allowed_files):
                runtime_failure(f'contains unrelated or symlinked data: {child}')
        for name in file_names:
            child = current_path / name
            relative = child.relative_to(source).as_posix()
            if child.is_symlink() or relative not in allowed_files:
                runtime_failure(f'contains unrelated or symlinked data: {child}')
    for entry in entries:
        relative = entry['path']
        path = package_path(source, relative)
        if not path.is_file() or path.stat().st_size != entry['size'] or sha256_file(path) != entry['sha256']:
            runtime_failure(f'has a size or SHA-256 mismatch for {relative}')
    try:
        if package_path(source, 'VERSION').read_text(encoding='utf-8').strip() != manifest['version']:
            runtime_failure('VERSION does not match the manifest version')
    except (OSError, UnicodeError) as exc:
        runtime_failure(f'cannot read VERSION ({exc})')
    for field, relative in (('lock_sha256', 'uv.lock'), ('template_sha256', RUNTIME_TEMPLATE_PATH)):
        if manifest.get(field) != entry_by_path[relative]['sha256']:
            runtime_failure(f'manifest {field} does not match {relative}')
    legal_files = manifest.get('legal_files')
    if not isinstance(legal_files, list) or any(safe_relative_path(path) not in entry_by_path for path in legal_files):
        runtime_failure('manifest legal_files is invalid')
    provenance = manifest.get('provenance')
    if not isinstance(provenance, dict) or any(provenance.get(field) != manifest.get(field) for field in RUNTIME_PROVENANCE_FIELDS):
        runtime_failure('manifest provenance does not match package metadata')
    if destination.is_symlink() or (destination.exists() and not destination.is_dir()):
        runtime_failure(f'local destination is not a directory: {destination}')
    if source.resolve() == destination.resolve() or source.resolve() in destination.resolve().parents or destination.resolve() in source.resolve().parents:
        runtime_failure('source and local destination must be separate directories')
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = Path(tempfile.mkdtemp(prefix='.AMADEUS.part-', dir=str(destination.parent)))
    backup = None
    try:
        for entry in entries:
            target = temporary.joinpath(*entry['path'].split('/'))
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(package_path(source, entry['path']), target)
        launcher_target = temporary.joinpath(*RUNTIME_NOTEBOOK_PATH.split('/'))
        launcher_target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(launcher_path, launcher_target)
        shutil.copy2(manifest_path, temporary / RUNTIME_MANIFEST_NAME)
        if destination.exists():
            backup = destination.parent / f'.{destination.name}.previous-{os.getpid()}'
            if backup.exists() or backup.is_symlink():
                runtime_failure(f'cannot create a safe backup beside {destination}')
            os.replace(destination, backup)
        try:
            os.replace(temporary, destination)
        except BaseException:
            if backup is not None and not destination.exists() and backup.exists():
                os.replace(backup, destination)
            raise
        temporary = None
        if backup is not None:
            shutil.rmtree(backup)
            backup = None
    finally:
        if temporary is not None and temporary.exists():
            shutil.rmtree(temporary, ignore_errors=True)
        if backup is not None and backup.exists() and not destination.exists():
            os.replace(backup, destination)
    return manifest

MANIFEST = validate_and_stage_runtime(RUNTIME_SOURCE, REPO_DIR)
print(f"AMADEUS package staged: version={MANIFEST['version']} source_ref={MANIFEST['source_ref']} source_revision={MANIFEST['source_revision']}")
print(f'Local package: {REPO_DIR}')

In [ ]:
import subprocess
import sys

# uv resolves uv.lock, so AMADEUS pins it; UV_VERSION ships with the runtime package.
UV_VERSION = (REPO_DIR / 'UV_VERSION').read_text(encoding='utf-8').strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'uv=={UV_VERSION}'], check=True)
reported = subprocess.run(['uv', '--version'], capture_output=True, text=True, check=True).stdout.split()[1]
if reported != UV_VERSION:
    raise RuntimeError(f'Expected uv {UV_VERSION} but found uv {reported}.')
subprocess.run(['uv', 'sync', '--frozen', '--extra', 'cu128'], cwd=str(REPO_DIR), check=True)
subprocess.run(['uv', 'run', '--no-sync', 'python', '-c', "import torch; print('Installed torch:', torch.__version__, 'CUDA:', torch.version.cuda, 'available:', torch.cuda.is_available())"], cwd=str(REPO_DIR), check=True)

In [ ]:
import subprocess

result = subprocess.run(
    ['uv', 'run', '--no-sync', 'python', 'tools/colab_runner.py', '--config', CONFIG_PATH],
    cwd=REPO_DIR,
    check=False,
)
if result.returncode != 0:
    raise RuntimeError(f'AMADEUS Colab workflow failed with exit code {result.returncode}. See the Drive logs.')
print('AMADEUS completed successfully.')
print(f'Results saved to: {CONFIG_PATH.rsplit("/", 1)[0]}')